# P4b — TF-IDF Models (M1 and M2)

## 1. Introduction and Objective

In Notebook 04a, the regex classifier achieved a validation F1 score of 0.6803 using only five manually selected keywords. Most of its predictive ability on the training set came from the phrase "back pain". I now examine whether learning lexical patterns directly from the training data can improve prediction beyond this manually defined baseline.

Drawing on the TF-IDF approach in Valmianski et al. (2019) and the L1-regularized logistic regression used by Miotto et al. (2020), I develop two supervised classifiers using word-level (M1) and character-level (M2) TF-IDF representations. The character-level representation may help capture spelling variations, abbreviations, and partial word patterns that word-level features may miss.

Both models use the same classifier and experimental partitions, allowing a comparison of the two text representations under a consistent modeling framework.

## 2. Development Dataset Preparation

I use the same modeling dataset and fixed patient-grouped partitions as in Notebook 04a to ensure consistency across models.

The training set is used to fit both the TF-IDF vectorizer and the logistic regression classifier. The validation set is used for regularization and classification threshold selection. All learned transformations are fitted on the training data only, and the test set remains held out until the final comparative evaluation.

In [1]:
from pathlib import Path
import pandas as pd

# Load the modeling dataset and fixed partition assignments
DATA_DIR = Path("../results/03_experimental_setup")

df = pd.read_csv(
    DATA_DIR / "modeling_dataset.csv",
    keep_default_na=False
)

splits = pd.read_csv(DATA_DIR / "dataset_splits.csv")

# Retain development partitions only
dev_splits = splits[splits["split"].isin(["train", "validation"])]

dev_df = df.merge(
    dev_splits,
    on="stay_id",
    how="inner",
    validate="one_to_one"
)

train_df = dev_df[dev_df["split"] == "train"].copy()
val_df = dev_df[dev_df["split"] == "validation"].copy()

# Prepare model inputs and reference labels
X_train = train_df["chiefcomplaint_clean"]
y_train = train_df["lbp_label"]

X_val = val_df["chiefcomplaint_clean"]
y_val = val_df["lbp_label"]

# Check input integrity
assert len(dev_df) == len(dev_splits)
assert dev_df["chiefcomplaint_clean"].notna().all()
assert dev_df["chiefcomplaint_clean"].str.strip().ne("").all()
assert dev_df["lbp_label"].isin([0, 1]).all()

# Summarize development partitions
summary = dev_df.groupby("split").agg(
    encounters=("stay_id", "size"),
    patients=("subject_id", "nunique"),
    positives=("lbp_label", "sum")
)

summary["negatives"] = summary["encounters"] - summary["positives"]
summary["prevalence (%)"] = (
    summary["positives"] / summary["encounters"] * 100
).round(2)

display(summary)

,encounters,patients,positives,negatives,prevalence (%)
split,,,,,
train,295877,143119,11429,284448,3.86
validation,42268,20483,1632,40636,3.86


## 3. Model Configuration

I use two TF-IDF representations: word-level unigrams and bigrams for M1, and character-level n-grams of length 2 to 5 for M2. Word-level features capture individual words and short phrases, while character-level features may capture spelling variations, abbreviations, and partial word patterns.

Both models use L1-regularized logistic regression with balanced class weights to account for class imbalance. I implement each model as a scikit-learn Pipeline so that the TF-IDF vectorizer and classifier are fitted together using only the training data.

The regularization parameter C and classification threshold will be jointly selected using validation F1 in Section 4.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Shared TF-IDF settings
tfidf_params = {
    "lowercase": True,
    "min_df": 1,
    "max_df": 1.0,
    "max_features": 50000,
    "norm": "l2",
    "use_idf": True,
    "smooth_idf": True
}

# Shared logistic regression settings
lr_params = {
    "l1_ratio": 1.0,
    "solver": "liblinear",
    "class_weight": "balanced",
    "max_iter": 1000,
    "random_state": 42
}

# M1: Word-level TF-IDF
m1_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        stop_words=None,
        **tfidf_params
    )),
    ("clf", LogisticRegression(**lr_params))
])

# M2: Character-level TF-IDF
m2_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5),
        **tfidf_params
    )),
    ("clf", LogisticRegression(**lr_params))
])

# Model selection candidates
C_GRID = [0.1, 1, 10]
THRESHOLD_GRID = [i / 100 for i in range(5, 100, 5)]

# Model registry
base_pipelines = {
    "M1": m1_pipeline,
    "M2": m2_pipeline
}

display(m1_pipeline)
display(m2_pipeline)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",50000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'char'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(2, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",50000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_

## 4. Model Training and Selection

For each model, I train three independent pipelines using C values of 0.1, 1, and 10. Each fitted pipeline is evaluated on the validation set across classification thresholds ranging from 0.05 to 0.95 in increments of 0.05.

I jointly select C and the classification threshold based on the highest positive-class validation F1 score. When F1 scores are tied within the same C setting, I prefer the higher threshold. When the best F1 scores are tied across different C values, I prefer the smaller C for stronger regularization.

The selected fitted pipeline is retained without refitting. This procedure is applied consistently to both M1 and M2 and can be reused in the training data efficiency experiment.

In [3]:
from sklearn.base import clone
from sklearn.metrics import f1_score
from time import perf_counter

def train_and_select(
    base_pipeline,
    X_train,
    y_train,
    X_val,
    y_val,
    c_grid,
    threshold_grid
):
    fitted_pipelines = {}
    records = []

    for c in c_grid:
        print(f"  Fitting C={c}...", flush=True)
        start = perf_counter()

        pipe = clone(base_pipeline)
        pipe.set_params(clf__C=c)
        pipe.fit(X_train, y_train)

        elapsed = perf_counter() - start
        print(f"  C={c} fitted in {elapsed:.1f} seconds.")

        fitted_pipelines[c] = pipe

        # Generate validation probabilities once per fitted pipeline
        val_prob = pipe.predict_proba(X_val)[:, 1]

        # Evaluate classification thresholds
        for threshold in threshold_grid:
            val_pred = (val_prob >= threshold).astype(int)

            records.append({
                "C": c,
                "threshold": threshold,
                "f1": f1_score(
                    y_val, val_pred, zero_division=0
                )
            })

    # Rank candidates using the predefined selection rules
    search_results = (
        pd.DataFrame(records)
        .sort_values(
            ["f1", "C", "threshold"],
            ascending=[False, True, False]
        )
        .reset_index(drop=True)
    )

    best = search_results.iloc[0]

    best_C = float(best["C"])
    best_threshold = float(best["threshold"])

    return {
        "pipeline": fitted_pipelines[best_C],
        "best_C": best_C,
        "best_threshold": best_threshold,
        "best_f1": float(best["f1"]),
        "search_results": search_results
    }

In [4]:
# Train and select both TF-IDF models
selected_models = {}

for model_id, base_pipeline in base_pipelines.items():
    print(f"Training {model_id}...")

    selected_models[model_id] = train_and_select(
        base_pipeline,
        X_train,
        y_train,
        X_val,
        y_val,
        C_GRID,
        THRESHOLD_GRID
    )

    print(f"{model_id} completed.")

Training M1...
  Fitting C=0.1...
  C=0.1 fitted in 1.5 seconds.
  Fitting C=1...
  C=1 fitted in 4.4 seconds.
  Fitting C=10...
  C=10 fitted in 953.8 seconds.
M1 completed.
Training M2...
  Fitting C=0.1...


/opt/anaconda3/envs/ed-lbp-nlp/lib/python3.12/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  C=0.1 fitted in 8.8 seconds.
  Fitting C=1...
  C=1 fitted in 19.8 seconds.
  Fitting C=10...
  C=10 fitted in 95.0 seconds.
M2 completed.


In [5]:
# Summarize the best threshold for each C
selection_tables = []

for model_id, result in selected_models.items():
    best_per_c = (
        result["search_results"]
        .drop_duplicates(subset="C")
        .sort_values("C")
        .copy()
    )

    best_per_c.insert(0, "model", model_id)

    selection_tables.append(best_per_c)

selection_summary = pd.concat(
    selection_tables, ignore_index=True
)

display(selection_summary.round(4))

# Summarize the selected model configurations
best_config_summary = pd.DataFrame([
    {
        "model": model_id,
        "best_C": result["best_C"],
        "best_threshold": result["best_threshold"],
        "validation_f1": result["best_f1"]
    }
    for model_id, result in selected_models.items()
]).set_index("model")

display(best_config_summary.round(4))

,model,C,threshold,f1
0,M1,0.1,0.90,0.6940
1,M1,1.0,0.85,0.6894
2,M1,10.0,0.90,0.6867
3,M2,0.1,0.90,0.6929
4,M2,1.0,0.90,0.6960
5,M2,10.0,0.90,0.6892


,best_C,best_threshold,validation_f1
model,,,
M1,0.1,0.9,0.694
M2,1.0,0.9,0.696


In [6]:
# Check convergence of the selected classifiers
for model_id, result in selected_models.items():
    clf = result["pipeline"].named_steps["clf"]

    print(
        f"{model_id}: "
        f"C={result['best_C']}, "
        f"n_iter={clf.n_iter_.max()}, "
        f"max_iter={clf.max_iter}"
    )

M1: C=0.1, n_iter=24, max_iter=1000
M2: C=1.0, n_iter=24, max_iter=1000


**Observation**

M1 achieves its highest validation F1 of 0.6940 with C=0.1, while M2 performs best with C=1.0 and a validation F1 of 0.6960. Both models select a classification threshold of 0.90, illustrating the importance of threshold selection when using balanced class weights in an imbalanced classification task.

Increasing C to 10 does not improve validation performance for either model and substantially increases computational cost. Although convergence warnings occur during the candidate search, both selected classifiers terminate after 24 iterations, well below the maximum of 1,000. I retain the selected pipelines and acknowledge incomplete convergence among the candidates as a limitation of the model selection process.

## 5. Validation Performance

I evaluate the selected M1 and M2 pipelines using their respective classification thresholds and compare their validation performance with the regex baseline from Notebook 04a.

In addition to precision, recall, F1, and confusion matrix counts, I report AUROC and Average Precision (AP) for the two TF-IDF models using their continuous prediction scores. These metrics are not reported for M0 because it produces only binary predictions.

Since the validation set is also used for model selection, these results represent development performance rather than an independent estimate of generalization.

In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Load M0 validation results
m0_metrics = pd.read_csv(
    "../results/04a_regex_baseline/m0_validation_metrics.csv"
)

m0_metrics["model"] = "M0"
m0_metrics["auroc"] = float("nan")
m0_metrics["ap"] = float("nan")

# Evaluate the selected TF-IDF models
records = []

for model_id, result in selected_models.items():
    pipe = result["pipeline"]
    threshold = result["best_threshold"]

    # Apply the selected validation threshold
    val_prob = pipe.predict_proba(X_val)[:, 1]
    val_pred = (val_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, val_pred, labels=[0, 1]
    ).ravel()

    records.append({
        "model": model_id,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "auroc": roc_auc_score(y_val, val_prob),
        "ap": average_precision_score(y_val, val_prob),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

tfidf_metrics = pd.DataFrame(records)

# Compare M0, M1, and M2
validation_summary = pd.concat(
    [m0_metrics, tfidf_metrics],
    ignore_index=True
).set_index("model")

# Display classification and ranking metrics
display(
    validation_summary[
        ["precision", "recall", "f1", "auroc", "ap"]
    ].round(4)
)

# Display confusion matrix counts
display(
    validation_summary[["tn", "fp", "fn", "tp"]]
)

,precision,recall,f1,auroc,ap
model,,,,,
M0,0.6686,0.6924,0.6803,NaN,NaN
M1,0.7145,0.6746,0.6940,0.9481,0.6591
M2,0.7079,0.6844,0.6960,0.9476,0.6653


,tn,fp,fn,tp
model,,,,
M0,40076,560,502,1130
M1,40196,440,531,1101
M2,40175,461,515,1117


**Observation**

Both TF-IDF models achieve modest improvements over the regex baseline, with validation F1 scores of 0.6940 for M1 and 0.6960 for M2, compared with 0.6803 for M0. Interestingly, the improvement comes primarily from higher precision rather than higher recall. Both models reduce false positives but identify slightly fewer positive encounters than the regex classifier.

The word-level and character-level representations perform similarly across F1, AUROC, and AP. Although character-level features may capture spelling variations and partial word patterns, the current results do not demonstrate a clear advantage over word-level features.

Combined with the findings from Notebook 04a, these results suggest that simple lexical signals already capture a substantial portion of the predictive information in ED chief complaints. However, these are development results obtained on the validation set, and the relative performance of the models remains to be confirmed on the held-out test set.

## 6. Save Results

Save the selected TF-IDF pipelines, model configurations, and validation results for subsequent evaluation. The selected C values and classification thresholds are stored explicitly to ensure consistent predictions during the final test evaluation.

In [8]:
import json
import joblib
import sklearn

OUTPUT_DIR = Path("../results/04b_tfidf_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the selected pipelines and configurations
for model_id, result in selected_models.items():
    prefix = model_id.lower()
    pipe = result["pipeline"]

    joblib.dump(
        pipe,
        OUTPUT_DIR / f"{prefix}_pipeline.joblib"
    )

    config = {
        "model_id": model_id,
        "best_C": result["best_C"],
        "best_threshold": result["best_threshold"],
        "validation_f1": result["best_f1"],
        "input_column": "chiefcomplaint_clean",
        "tfidf_params": pipe.named_steps["tfidf"].get_params(),
        "classifier_params": pipe.named_steps["clf"].get_params(),
        "sklearn_version": sklearn.__version__
    }

    with open(OUTPUT_DIR / f"{prefix}_config.json", "w") as f:
        json.dump(config, f, indent=4, default=str)

    # Save model selection results
    result["search_results"].to_csv(
        OUTPUT_DIR / f"{prefix}_selection_results.csv",
        index=False
    )

# Save validation performance comparison
validation_summary.to_csv(
    OUTPUT_DIR / "validation_metrics.csv"
)

print("M1 and M2 pipelines, configurations, and results saved.")

M1 and M2 pipelines, configurations, and results saved.
